<a href="https://www.kaggle.com/code/sreyaroychowdhury/snntorch?scriptVersionId=327433050" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install snntorch -q

import os
import glob
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import snntorch as snn
import numpy as np

# Configurations matching the 1st Order LIF paper requirements
IMG_SIZE     = 64
NUM_STEPS    = 25
BATCH_SIZE   = 128
EPOCHS       = 50
LR           = 5e-4
BETA         = 0.9
THRESHOLD    = 1.0
DROPOUT_P    = 0.5
HIDDEN_SIZE  = 512
NUM_CLASSES  = 2
SEED         = 42

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Verify Device (Should say cuda):', device)

In [ ]:
class DroNetCollisionDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []

        for seq_dir in sorted(glob.glob(os.path.join(root, '*'))):
            if not os.path.isdir(seq_dir):
                continue
            label_file = os.path.join(seq_dir, 'labels.txt')
            img_dir    = os.path.join(seq_dir, 'images')
            if not os.path.isfile(label_file) or not os.path.isdir(img_dir):
                continue

            with open(label_file, 'r') as f:
                labels = [int(l.strip()) for l in f if l.strip() != '']

            img_paths = sorted(
                glob.glob(os.path.join(img_dir, '*.jpg')) +
                glob.glob(os.path.join(img_dir, '*.png'))
            )

            for img_path, label in zip(img_paths, labels):
                self.samples.append((img_path, label))

        print(f'Loaded {len(self.samples)} samples from {root}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

# Transform: Scales pixels strictly to [0, 1] for Bernoulli spike generation probabilities
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor() 
])

BASE = '/kaggle/input/datasets/sreyaroychowdhury/dronet-collision/collision_dataset'

train_dataset = DroNetCollisionDataset(os.path.join(BASE, 'training'), transform)
test_dataset  = DroNetCollisionDataset(os.path.join(BASE, 'testing'), transform)

# Inverse frequency calculation to balance out your dataset metrics
num_neg = 24968
num_pos = 4894
total = num_neg + num_pos

weight_neg = total / (2.0 * num_neg)
weight_pos = total / (2.0 * num_pos)
class_weights = torch.FloatTensor([weight_neg, weight_pos]).to(device)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
class FirstOrderLIFSNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input Layer: 4096 flattened vector -> 512 Hidden Neurons
        self.fc1 = nn.Linear(IMG_SIZE * IMG_SIZE, HIDDEN_SIZE)
        self.lif1 = snn.Leaky(beta=BETA, threshold=THRESHOLD, learn_beta=True, learn_threshold=True)
        self.dropout = nn.Dropout(DROPOUT_P)
        
        # Output Layer: 512 Hidden Neurons -> 2 Output Classes
        self.fc2 = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)
        self.lif2 = snn.Leaky(beta=BETA, threshold=THRESHOLD, learn_beta=True, learn_threshold=True)

    def forward(self, x):
        # Let snntorch naturally match standard tensor allocation on the individual active GPU
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        
        spk2_rec = []
        x_flattened = x.view(x.size(0), -1)
        
        # Run temporal dynamics loops over 25 time steps
        for step in range(NUM_STEPS):
            spike_in = torch.bernoulli(x_flattened)
            
            cur1 = self.fc1(spike_in)
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1 = self.dropout(spk1)
            
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            spk2_rec.append(spk2)
            
        return torch.stack(spk2_rec, dim=0)

# Load directly to the verified device memory space
model = FirstOrderLIFSNN().to(device)
print("Model loaded onto hardware instance safely.")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

import snntorch.functional as sf
loss_fn = sf.ce_count_loss(weight=class_weights)

In [ ]:
print("Beginning SNN training loop...")
for epoch in range(EPOCHS):
    model.train()
    loss_hist = []
    correct_count = 0
    total_count = 0
    
    for data, targets in train_loader:
        data, targets = data.to(device), targets.to(device)
        
        optimizer.zero_grad()
        spk_rec = model(data)  # Output: [25, batch_size, 2]
        
        loss_val = loss_fn(spk_rec, targets)
        loss_val.backward()
        optimizer.step()
        
        loss_hist.append(loss_val.item())
        
        # Pull prediction targets from spike frequencies
        _, idx = spk_rec.sum(dim=0).max(1)
        correct_count += (idx == targets).sum().item()
        total_count += targets.size(0)
        
    train_acc = (correct_count / total_count) * 100
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {np.mean(loss_hist):.4f} | Train Acc: {train_acc:.2f}%")

In [ ]:
def float_to_q115_hex(tensor, filename):
    flat_weights = tensor.detach().cpu().numpy().flatten()
    
    # Clip parameters tightly to ensure no two's complement sign inversion
    clipped_weights = np.clip(flat_weights, -1.0, 32767.0 / 32768.0)
    
    # Scale up by 2^15 factor representation scale
    q115_int = np.round(clipped_weights * 32768).astype(np.int32)
    
    with open(filename, 'w') as f:
        for val in q115_int:
            if val < 0:
                val = (1 << 16) + val
            f.write(f"{val:04X}\n")
            
    print(f"Successfully generated '{filename}' ({len(q115_int)} parameters).")

float_to_q115_hex(model.fc1.weight, "layer1_weights_hex.txt")
float_to_q115_hex(model.fc2.weight, "layer2_weights_hex.txt")